
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 03: Optimizing Apache Spark

This notebook demonstrates key performance optimization techniques for Apache Spark applications using the TPC-H dataset:

1. Understanding Spark Performance - Resource utilization and data flow patterns
2. Partitioning Strategies - DataFrame partitioning for better distribution
3. Reducing Shuffle Operations - Minimizing expensive shuffle operations
4. DataFrame Caching - Effective data persistence strategies
5. Query Optimization - Understanding Catalyst optimizer and execution plans

Throughout this notebook, we'll examine the Spark UI at each step to understand the impact of various optimizations.

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.
<br></br>

```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run  ./Includes/Classroom-Setup-Common

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Course Catalog:,
Your Schema:,


## B. Data Setup

First, let's copy some tables from the TPC-H sample dataset to our own catalog/schema for our optimization demonstrations and look at some configuration items.

In [0]:
# make copies of "lineitem", "orders" tables

for table in ["lineitem", "orders"]:
    print(f"Creating local copy of {table}...")
    
    # Create a local copy
    spark.table(f"samples.tpch.{table}").write.mode("overwrite").saveAsTable(table)

Creating local copy of lineitem...
Creating local copy of orders...


In [0]:
orders_df = spark.table("orders")
lineitems_df = spark.table("lineitem")

In [0]:
from pyspark.sql.functions import col, sum, count, avg
import time

# Get cluster info for better partitioning
num_cores = sc.defaultParallelism
print(f"Default parallelism (cores): {num_cores}")

# Get shuffle partitions to 200 (Spark default)
shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Default shuffle partitions: {shuffle_partitions}")

Default parallelism (cores): 4
Default shuffle partitions: 200


## C. Understanding Partitioning and Shuffling

Let's deep dive into partitioning and understand how different operations affect the numbers or sizes of partitions and shuffling.

###1. Default Partitioning
The default number of partitions in an input dataframe (read using the `DataFrameReader` from files or from a table) is equivalent to the number of files in the dataset

In [0]:
# Note that the default number of partitions in the dataframe is equivalent to the number of files in the dataset
print(f"Default partitions (lineitem): {lineitems_df.rdd.getNumPartitions()}")
display(spark.sql("DESCRIBE DETAIL lineitem").select("numFiles"))

Default partitions (lineitem): 10


numFiles
10


###2. Narrow Transformations and Partition Counts
Narrow transformations (such as `filter`, `select`, `drop`, `withColumn`) will either retain the same number of partitions or reduce the number of partitions in the resultant dataframe, let's have a look.

In [0]:
print(f"Starting number of partitions: {lineitems_df.rdd.getNumPartitions()}")
high_value_lineitems_df = lineitems_df.filter("l_extendedprice > 60000 AND l_linenumber < 2")
print(f"Number of partitions (after narrow transformation): {high_value_lineitems_df.rdd.getNumPartitions()}")

Starting number of partitions: 10
Number of partitions (after narrow transformation): 10


###3. Narrow Transformations and Skew
Narrow transformations such as `filter` can create uneven partition sizes (as filtered records may not be distributed equally), to demonstrate this we will write out the results of our dataframe and look at the resultant file sizes.

In [0]:
def show_partition_sizes(input_dataframe):
    # Define the output directory
    output_path = f"{DA.catalog_name}/{DA.schema_name}/high_value_lineitems"

    # Remove the directory if it already exists
    dbutils.fs.rm(output_path, True)

    # Write the DataFrame as parquet files
    input_dataframe.write \
        .format("parquet") \
        .option("compression", "none") \
        .mode("overwrite") \
        .save(output_path)

    # List the files in the directory
    print("Files in the output directory:")
    files = dbutils.fs.ls(output_path)
    for i, file in enumerate([f for f in files if f.path.endswith(".parquet")]):
        print(f"Partition {i}: Size: {file.size} bytes")

show_partition_sizes(high_value_lineitems_df)

Files in the output directory:
Partition 0: Size: 12407225 bytes
Partition 1: Size: 12388459 bytes
Partition 2: Size: 12356286 bytes
Partition 3: Size: 12396301 bytes
Partition 4: Size: 12368012 bytes
Partition 5: Size: 12385285 bytes
Partition 6: Size: 6917398 bytes
Partition 7: Size: 6847496 bytes
Partition 8: Size: 6789332 bytes
Partition 9: Size: 7480999 bytes


In [0]:
# To normalize skew, lets repartition
balanced_df = high_value_lineitems_df.repartition(10)
show_partition_sizes(balanced_df)

# Alternatively you could repartition by a specific column (or columns) for better data distribution
# balanced_df = high_value_lineitems_df.repartition(10, "l_orderkey")
# show_partition_sizes(balanced_df)

Files in the output directory:
Partition 0: Size: 10233750 bytes
Partition 1: Size: 10237570 bytes
Partition 2: Size: 10236093 bytes
Partition 3: Size: 10237913 bytes
Partition 4: Size: 10232219 bytes
Partition 5: Size: 10230247 bytes
Partition 6: Size: 10231662 bytes
Partition 7: Size: 10230489 bytes
Partition 8: Size: 10236136 bytes
Partition 9: Size: 10236008 bytes


###4. Analyzing Shuffling
Look at the Spark UI for the job above☝️.  Notice that the above operation forced 90% of the dataset to shuffle (moving data between partitions).

In [0]:
# Use coalesce to reduce partitions without full shuffle, this may be better when you just need fewer partitions and don't mind some imbalance
# Note: Coalesce does not address skew directly however it will consolidate partitions
smaller_df = high_value_lineitems_df.coalesce(5)
show_partition_sizes(smaller_df)

Files in the output directory:
Partition 0: Size: 24761016 bytes
Partition 1: Size: 24717940 bytes
Partition 2: Size: 24718635 bytes
Partition 3: Size: 13730215 bytes
Partition 4: Size: 14235588 bytes


Look at the Spark UI again now for the job above☝️.  Notice that the above operation did not result in shuffling.

###5. Wide Transformations and Partitioning

Wide transformations (like `groupBy`, `join`, `repartition`) cause data to be shuffled across the network. They affect partitioning in significant ways:

1. They typically change the number of partitions (based on spark.sql.shuffle.partitions)
2. They redistribute data across partitions based on keys
3. They can create or resolve data skew depending on how they're used


In [0]:
# Check partitions in original DataFrame
print(f"Original partitions in lineitems_df: {lineitems_df.rdd.getNumPartitions()}")
# Apply a groupBy (wide transformation)
grouped_lineitems = lineitems_df.groupBy("l_suppkey").count()
print(f"Partitions after groupBy: {grouped_lineitems.rdd.getNumPartitions()}")
# The resultant number of partitions is determined by AQE (adaptive query execution)

Original partitions in lineitems_df: 10
Partitions after groupBy: 4


## D. Understanding Shuffle Partitions and AQE

Let's demonstrate the effects of the `spark.sql.shuffle.partitions` configuration setting and Adaptive Query Execution (AQE).

In [0]:
# Force disable AQE for a test
spark.conf.set("spark.sql.adaptive.enabled", "false")
grouped_lineitems_no_aqe = lineitems_df.groupBy("l_suppkey").count()
print(f"Partitions after groupBy with AQE disabled: {grouped_lineitems_no_aqe.rdd.getNumPartitions()}")

# Why 200? remember the default value of spark.sql.shuffle.partitions

Partitions after groupBy with AQE disabled: 200


In [0]:
# Let's set the number of shuffle partitions to the number of cores
spark.conf.set("spark.sql.shuffle.partitions", num_cores)
grouped_lineitems_no_aqe_updated_shuffle_parts = lineitems_df.groupBy("l_suppkey").count()
print(f"Partitions after groupBy with AQE disabled and shuffle partitions set: {grouped_lineitems_no_aqe_updated_shuffle_parts.rdd.getNumPartitions()}")

Partitions after groupBy with AQE disabled and shuffle partitions set: 4


In [0]:
# Let's re-enable AQE now and re-set the number of shuffle partitions to the default
spark.conf.set("spark.sql.shuffle.partitions", 200)
spark.conf.set("spark.sql.adaptive.enabled", "true")

## E. Analyzing Explain Plans

Spark's `explain()` function is a powerful tool for understanding query execution. It shows how Spark's Catalyst optimizer transforms your code into execution steps.

Let's examine explain plans for different operations to better understand optimization opportunities:

In [0]:
# Simple query explain plan
simple_query = lineitems_df.filter("l_shipdate > '1995-01-01'").groupBy("l_shipmode").count()

print("SIMPLE QUERY EXPLAIN:")
simple_query.explain()

SIMPLE QUERY EXPLAIN:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   HashAggregate(keys=[l_shipmode#25377], functions=[finalmerge_count(merge count#25920L) AS count(1)#25915L])
   +- Exchange hashpartitioning(l_shipmode#25377, 200), ENSURE_REQUIREMENTS, [plan_id=18109]
      +- HashAggregate(keys=[l_shipmode#25377], functions=[partial_count(1) AS count#25920L])
         +- Project [l_shipmode#25377]
            +- Filter ((if (isnotnull(_databricks_internal_edge_computed_column_skip_row#25934)) (_databricks_internal_edge_computed_column_skip_row#25934 = false) else isnotnull(raise_error(DELTA_SKIP_ROW_COLUMN_NOT_FILLED, map(keys: [], values: []), NullType)) AND isnotnull(l_shipdate#25373)) AND (l_shipdate#25373 > 1995-01-01))
               +- FileScan parquet dbacademy.labuser10806356_1751473610.lineitem[l_shipdate#25373,l_shipmode#25377,_databricks_internal_edge_computed_column_skip_row#25934] Batched: true, DataFilters: [isnotnull(l_shipdate#25373),

The following query has several inefficiencies as it is written, let's look at how the optimizer handles this.

In [0]:
inefficient_query = (
    lineitems_df
    # Filter applied late in the transformation
    .select("l_orderkey", "l_shipdate", "l_shipmode", "l_extendedprice", "l_discount")
    # Join before filtering (inefficient)
    .join(
        orders_df.select("o_orderkey", "o_orderdate", "o_orderpriority"),
        lineitems_df["l_orderkey"] == orders_df["o_orderkey"]
    )
    # Filters that could be pushed down before the join
    .filter(col("l_shipdate") > "1995-01-01")
    .filter(col("o_orderdate") > "1995-01-01")
    # Late filter on shipmode
    .filter(col("l_shipmode").isin("AIR", "MAIL"))
    # Calculate revenue
    .withColumn("revenue", col("l_extendedprice") * (1 - col("l_discount")))
    # Group and aggregate
    .groupBy("l_shipmode", "o_orderpriority")
    .agg(
        sum("revenue").alias("total_revenue"),
        count("*").alias("order_count")
    )
)

In [0]:
# Show the logical plan (what was written)
print("INEFFICIENT QUERY PLAN:")
inefficient_query.explain(mode="formatted")

INEFFICIENT QUERY PLAN:
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- SortMergeJoin Inner (11)
               :- Sort (5)
               :  +- Exchange (4)
               :     +- Project (3)
               :        +- Filter (2)
               :           +- Scan parquet dbacademy.labuser10806356_1751473610.lineitem (1)
               +- Sort (10)
                  +- Exchange (9)
                     +- Project (8)
                        +- Filter (7)
                           +- Scan parquet dbacademy.labuser10806356_1751473610.orders (6)


(1) Scan parquet dbacademy.labuser10806356_1751473610.lineitem
Output [6]: [l_orderkey#25363L, l_extendedprice#25368, l_discount#25369, l_shipdate#25373, l_shipmode#25377, _databricks_internal_edge_computed_column_skip_row#26147]
Batched: true
Location: PreparedDeltaFileIndex [s3://unity-catalogs-us-west-2/metastore/4

In [0]:
# Show the optimized physical plan (what Spark will actually execute)
print("\nPHYSICAL PLAN (what Spark optimizes it to):")
inefficient_query.explain(mode="extended")


PHYSICAL PLAN (what Spark optimizes it to):
== Parsed Logical Plan ==
'Aggregate ['l_shipmode, 'o_orderpriority], ['l_shipmode, 'o_orderpriority, 'sum('revenue) AS total_revenue#25984, 'count(1) AS order_count#25985]
+- Project [l_orderkey#25363L, l_shipdate#25373, l_shipmode#25377, l_extendedprice#25368, l_discount#25369, o_orderkey#25329L, o_orderdate#25333, o_orderpriority#25334, (l_extendedprice#25368 * (cast(1 as decimal(1,0)) - l_discount#25369)) AS revenue#25965]
   +- Filter l_shipmode#25377 IN (AIR,MAIL)
      +- Filter (o_orderdate#25333 > cast(1995-01-01 as date))
         +- Filter (l_shipdate#25373 > cast(1995-01-01 as date))
            +- Join Inner, (l_orderkey#25363L = o_orderkey#25329L)
               :- Project [l_orderkey#25363L, l_shipdate#25373, l_shipmode#25377, l_extendedprice#25368, l_discount#25369]
               :  +- SubqueryAlias dbacademy.labuser10806356_1751473610.lineitem
               :     +- Relation dbacademy.labuser10806356_1751473610.lineitem[l_

Looking at these two execution plans, we can see how Spark optimized the inefficient query:

1. **Filter Pushdown**: In the optimized plan, notice how the filters were pushed down to the scan operations:
   ```
   +- PhotonScan parquet ...lineitem
      DictionaryFilters: [(l_shipdate#69 > 1995-01-01), l_shipmode#73 IN (AIR,MAIL)]
   ```
   Even though we placed these filters after the join in our query, Spark moved them to the earliest possible point (during the initial data read).

2. **Column Pruning**: The optimizer only reads the columns it actually needs:
   ```
   ReadSchema: struct<l_orderkey:bigint,l_extendedprice:decimal(18,2),l_discount:decimal(18,2),l_shipdate:date,l_shipmode:string>
   ```
   Even though we selected more columns in our initial query, Spark only reads what's necessary for the final result.

3. **Filter Combination**: In the optimized logical plan, you can see how multiple separate filters have been combined:
   ```
   +- Filter (((isnotnull(l_shipdate#69) AND isnotnull(l_orderkey#59L)) AND (l_shipdate#69 > 1995-01-01)) AND l_shipmode#73 IN (AIR,MAIL))
   ```
   All the filter conditions were combined into a single operation.

4. **Projection Optimization**: The execution only projects necessary columns at each step.

### The Bottom Line

Spark's optimizer transformed this into a much more efficient plan that:
1. Filters data as early as possible
2. Reads only necessary columns
3. Combines multiple filter conditions
4. Uses a more efficient ordering of operations

> NOTE: this doesn't mean you shouldn't write queries as you would expect them to be executed!

## F. DataFrame Caching

Caching can significantly improve performance for iterative operations on the same data. Let's demonstrate its impact.

First, let's run a sequence of operations without caching:

**👀 Spark UI Observation:** Take note of how each query has to read from the source tables repeatedly.
- Look at the "Input" metrics that show how much data is read
- Notice the full execution plan for each job


In [0]:
from pyspark.sql.functions import avg, max

# Uncached query
high_value_line_items_df = lineitems_df.filter("l_extendedprice > 100000").select(
    "l_orderkey", "l_shipdate", "l_shipmode", "l_extendedprice", "l_discount"
)
print(f"There are a total of {high_value_line_items_df.count()} high value line items")

avg_price_by_ship_mode = high_value_line_items_df.groupBy("l_shipmode").agg(
    avg("l_extendedprice").alias("avg_price")
)
print(f"Average price by ship mode (high_value_line_items_df is re-evaluated):")
display(avg_price_by_ship_mode)

max_price_by_ship_mode = high_value_line_items_df.groupBy("l_shipmode").agg(
    max("l_extendedprice").alias("max_price")
)
print(f"Max price by ship mode (high_value_line_items_df is re-evaluated again):")
display(max_price_by_ship_mode)

There are a total of 20382 high value line items
Average price by ship mode (high_value_line_items_df is re-evaluated):


l_shipmode,avg_price
AIR,101468.092892
MAIL,101496.225559
RAIL,101457.322949
SHIP,101462.146887
TRUCK,101466.737948
REG AIR,101444.967859
FOB,101456.673800


Max price by ship mode (high_value_line_items_df is re-evaluated again):


l_shipmode,max_price
AIR,104949.50
MAIL,104948.00
RAIL,104899.50
SHIP,104848.00
TRUCK,104899.50
REG AIR,104849.00
FOB,104899.50


In [0]:
# Cached query
high_value_line_items_df =  lineitems_df.filter("l_extendedprice > 100000").select("l_orderkey", "l_shipdate", "l_shipmode", "l_extendedprice", "l_discount")
high_value_line_items_df.cache()
print(f"There are a total of {high_value_line_items_df.count()} high value line items")

avg_price_by_ship_mode = high_value_line_items_df.groupBy("l_shipmode").agg(avg("l_extendedprice").alias("avg_price"))
print(f"Average price by ship mode (high_value_line_items_df is NOT re-evaluated):")
avg_price_by_ship_mode.show()

max_price_by_ship_mode = high_value_line_items_df.groupBy("l_shipmode").agg(max("l_extendedprice").alias("max_price"))
print(f"Max price by ship mode (high_value_line_items_df is NOT re-evaluated):")
max_price_by_ship_mode.show()

There are a total of 20382 high value line items
Average price by ship mode (high_value_line_items_df is NOT re-evaluated):
+----------+-------------+
|l_shipmode|    avg_price|
+----------+-------------+
|       AIR|101468.092892|
|      MAIL|101496.225559|
|      RAIL|101457.322949|
|      SHIP|101462.146887|
|     TRUCK|101466.737948|
|   REG AIR|101444.967859|
|       FOB|101456.673800|
+----------+-------------+

Max price by ship mode (high_value_line_items_df is NOT re-evaluated):
+----------+---------+
|l_shipmode|max_price|
+----------+---------+
|       AIR|104949.50|
|      MAIL|104948.00|
|      RAIL|104899.50|
|      SHIP|104848.00|
|     TRUCK|104899.50|
|   REG AIR|104849.00|
|       FOB|104899.50|
+----------+---------+



In [0]:
# un-cache the dataframe
lineitems_df.unpersist()

DataFrame[l_orderkey: bigint, l_partkey: bigint, l_suppkey: bigint, l_linenumber: int, l_quantity: decimal(18,2), l_extendedprice: decimal(18,2), l_discount: decimal(18,2), l_tax: decimal(18,2), l_returnflag: string, l_linestatus: string, l_shipdate: date, l_commitdate: date, l_receiptdate: date, l_shipinstruct: string, l_shipmode: string, l_comment: string]

## Clean Up

Let's clean up the resources we created for this demo.


In [0]:
# Drop the tables we created
for table in ["lineitem", "orders"]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")

print("Clean up completed!")

Clean up completed!


## Key Takeaways

1. **Understanding Spark Performance**
   - Monitor resource utilization, data flow patterns, and bottlenecks using the Spark UI
   - Understand the impact of data size, formats, and distribution on performance

2. **Partitioning Strategies**
   - Choose high-cardinality columns for even distribution
   - Target 100-200MB per partition as a general guideline
   - Use `repartition()` and `coalesce()` to control partition count

3. **Reducing Shuffle Operations**
   - Filter early to reduce data volume
   - Use broadcast joins for small tables
   - Maintain consistent partitioning where possible
   - Monitor shuffle spill metrics (memory vs disk)

4. **DataFrame Caching**
   - Cache DataFrames that are reused in multiple operations
   - Use `cache()` or `persist()` explicitly
   - Remember to `unpersist()` when data is no longer needed
   
5. **Query Optimization**
   - Understand Catalyst optimizer and execution plans with `explain()`
   - Leverage predicate pushdown and column pruning
   - Use the appropriate join strategy for your data

6. **Predictive Optimization with Delta Lake**
   - Delta Lake's **Predictive Optimization** automatically leverages your `OPTIMIZE` and Z-ORDER operations
   - When you run queries, the query optimizer automatically considers your Z-ORDER indexes
   - Benefits:
     - No need to explicitly hint which indexes to use in your queries
     - The system automatically prunes files based on Z-ORDER columns
     - Queries are automatically optimized for better performance
     - Reduces I/O by skipping files that don't contain relevant data
   - Example: If you've run `OPTIMIZE table ZORDER BY (date_col, region)`:
     - A query with `WHERE date_col = '2023-01-01' AND region = 'APAC'` automatically benefits
     - The optimizer uses Z-ORDER statistics to read only relevant files
     - This happens transparently without any special query modifications

7. **Best Practices**
   - Use the Spark UI to monitor performance
   - Filter early and select only needed columns
   - Optimize join operations and join order
   - Minimize UDFs in favor of built-in functions



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
